# 03 — Modeling & Validation

Trains baseline models (Linear, Ridge, Lasso regression), evaluates via chronological split and walk-forward validation, and performs feature ablation.

**Key principle**: Chronological validation only — no random splits.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.utils.paths import load_config
from src.data.load_data import load_processed
from src.validation.walk_forward import chronological_split, run_walk_forward_grid
from src.models.train_model import run_full_model_grid, train_and_evaluate
from src.models.model_utils import get_feature_columns

config = load_config()
sns.set_theme(style='whitegrid', font_scale=1.1)

## 1. Chronological Split

In [ ]:
datasets = {}
splits = {}
for sym in config['assets']:
    df = load_processed(sym, 'features')
    datasets[sym] = df
    train, val, test = chronological_split(df)
    splits[sym] = (train, val, test)
    print(f"{sym}:")
    print(f"  Train: {len(train):,} rows ({train['timestamp'].min()} → {train['timestamp'].max()})")
    print(f"  Val:   {len(val):,} rows")
    print(f"  Test:  {len(test):,} rows ({test['timestamp'].min()} → {test['timestamp'].max()})")
    print()

## 2. Model Comparison — Fixed Split

In [ ]:
for sym in config['assets']:
    train, val, test = splits[sym]
    comparison = run_full_model_grid(train, test,
                                      horizons=['1s', '5s', '10s', '30s'])
    print(f"\n{'='*60}")
    print(f"  {sym} — Model Comparison (test set)")
    print(f"{'='*60}")
    cols = [c for c in ['model', 'target', 'pearson_ic', 'spearman_ic', 'mse', 'decile_spread']
            if c in comparison.columns]
    display(comparison[cols].round(4))

## 3. Feature Ablation

In [ ]:
for sym in config['assets']:
    train, val, test = splits[sym]
    ablation = run_full_model_grid(train, test, horizons=['5s'],
        feature_sets=['orderbook_only', 'trade_only', 'return_vol_only', 'all'])
    
    print(f"\n{sym} — Feature Ablation (5s horizon)")
    cols = [c for c in ['model', 'feature_set', 'pearson_ic', 'spearman_ic', 'decile_spread']
            if c in ablation.columns]
    display(ablation[cols].round(4))

In [ ]:
# Visualize ablation for Ridge
for sym in config['assets']:
    train, val, test = splits[sym]
    ablation = run_full_model_grid(train, test, horizons=['5s'],
        feature_sets=['orderbook_only', 'trade_only', 'return_vol_only', 'all'])
    
    ridge_abl = ablation[ablation['model'] == 'ridge'].copy()
    if not ridge_abl.empty and 'pearson_ic' in ridge_abl.columns:
        fig, ax = plt.subplots(figsize=(8, 5))
        bars = ax.bar(ridge_abl['feature_set'], ridge_abl['pearson_ic'],
                      color=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6'])
        ax.set_title(f'{sym} — Ridge IC by Feature Group (5s)', fontsize=14)
        ax.set_ylabel('Pearson IC')
        ax.set_xlabel('Feature Set')
        for bar, val in zip(bars, ridge_abl['pearson_ic']):
            if not np.isnan(val):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                        f'{val:.3f}', ha='center', fontsize=11)
        plt.tight_layout()
        plt.show()

## 4. Walk-Forward Validation

In [ ]:
for sym in config['assets']:
    df = datasets[sym]
    wf = run_walk_forward_grid(df, horizons=['1s', '5s', '10s', '30s'],
                                model_names=['linear', 'ridge', 'lasso'], n_folds=5)
    print(f"\n{'='*60}")
    print(f"  {sym} — Walk-Forward Validation")
    print(f"{'='*60}")
    cols = [c for c in ['model', 'horizon', 'mean_pearson_ic', 'std_pearson_ic',
                        'ic_tstat', 'mean_spearman_ic', 'n_folds', 'total_test_rows']
            if c in wf.columns]
    display(wf[cols].round(4))

In [ ]:
# Visualize walk-forward IC by horizon (Ridge only)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, sym in zip(axes, config['assets']):
    df = datasets[sym]
    wf = run_walk_forward_grid(df, horizons=['1s', '5s', '10s', '30s'],
                                model_names=['ridge'], n_folds=5)
    if not wf.empty:
        ax.bar(wf['horizon'], wf['mean_pearson_ic'],
               yerr=wf['std_pearson_ic'], capsize=4,
               color='#3498db', alpha=0.8)
        ax.set_title(f'{sym} — Walk-Forward IC (Ridge)', fontsize=13)
        ax.set_xlabel('Horizon')
        ax.set_ylabel('Mean Pearson IC')
        ax.axhline(0, color='black', linewidth=0.8)
        for i, row in wf.iterrows():
            ax.text(i, row['mean_pearson_ic'] + row['std_pearson_ic'] + 0.01,
                    f"t={row['ic_tstat']:.1f}", ha='center', fontsize=9)

plt.suptitle('Walk-Forward Validation: IC by Horizon', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Feature Importance (Ridge Coefficients)

In [ ]:
from src.models.model_utils import extract_feature_importance

for sym in config['assets']:
    train, val, test = splits[sym]
    result = train_and_evaluate(train, test, 'y_return_5s', 'ridge', 'all', 'regression')
    
    if 'pipeline' in result:
        fi = extract_feature_importance(result['pipeline'], result.get('feature_names', []))
        fi = fi.sort_values('importance', ascending=True).tail(20)
        
        fig, ax = plt.subplots(figsize=(10, 8))
        colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in fi['importance']]
        ax.barh(fi['feature'], fi['importance'], color=colors)
        ax.set_title(f'{sym} — Top 20 Ridge Coefficients (5s target)', fontsize=13)
        ax.set_xlabel('Standardized Coefficient')
        ax.axvline(0, color='black', linewidth=0.8)
        plt.tight_layout()
        plt.show()